# Data Cleaning: Order Date Standardization
This notebook merges yearly files (if needed) and standardizes `order_date` to `YYYY-MM-DD`.

In [21]:
from pathlib import Path
import pandas as pd

base_dir = Path.cwd().parent
raw_dir = (base_dir / "data" / "raw").resolve()
cleaned_dir = (base_dir / "data" / "cleaned").resolve()
cleaned_dir.mkdir(parents=True, exist_ok=True)

merged_path = raw_dir / "amazon_india_all_years.csv"
if not merged_path.exists():
    raise FileNotFoundError(f"Merged dataset not found: {merged_path}")

merged_df = pd.read_csv(merged_path)
print(f"Loaded merged dataset: {merged_path}")

Loaded merged dataset: C:\Users\admin\Desktop\Amazon_India_Sales_Analytics\data\raw\amazon_india_all_years.csv


In [22]:
# Keep a raw copy for cleaning workflows
raw_df = merged_df.copy()
print(f"Raw copy ready: {raw_df.shape}")

Raw copy ready: (1127609, 34)


In [30]:
# Load latest cleaned dataset if present to skip re-cleaning
cleaned_parquet = cleaned_dir / "amazon_india_all_years_cleaned.parquet"
cleaned_csv = cleaned_dir / "amazon_india_all_years_cleaned.csv"
cleaned_df_loaded = False

if cleaned_parquet.exists():
    cleaned_df = pd.read_parquet(cleaned_parquet)
    cleaned_df_loaded = True
    print(f"Loaded cleaned dataset: {cleaned_parquet}")
elif cleaned_csv.exists():
    cleaned_df = pd.read_csv(cleaned_csv)
    cleaned_df_loaded = True
    print(f"Loaded cleaned dataset: {cleaned_csv}")
else:
    print("No cleaned dataset found yet; will build from raw.")


Loaded cleaned dataset: C:\Users\admin\Desktop\Amazon_India_Sales_Analytics\data\cleaned\amazon_india_all_years_cleaned.csv


In [24]:
 # Helper: save cleaned dataset after each column cleanup (CSV only)
def save_cleaned_df(cleaned_df, cleaned_dir, note=None):
    cleaned_path_csv = cleaned_dir / "amazon_india_all_years_cleaned.csv"
    cleaned_df.to_csv(cleaned_path_csv, index=False)
    print(f"Saved cleaned dataset to: {cleaned_path_csv}")
    if note:
        print(f"Checkpoint: {note}")


In [16]:
# Clean and standardize order_date to YYYY-MM-DD
if "order_date" not in raw_df.columns:
    raise KeyError("order_date column not found in merged dataset.")

if not cleaned_df_loaded:
    cleaned_df = raw_df.copy()

if not pd.api.types.is_datetime64_any_dtype(cleaned_df["order_date"]):
    cleaned_df["order_date"] = pd.to_datetime(
        cleaned_df["order_date"], errors="coerce", format="mixed"
    )

save_cleaned_df(cleaned_df, cleaned_dir, note="order_date cleaned")

Skipped parquet export due to error: A type extension with name pandas.period already defined
Saved cleaned dataset to: C:\Users\admin\Desktop\Amazon_India_Sales_Analytics\data\cleaned\amazon_india_all_years_cleaned.csv
Checkpoint: order_date cleaned


In [17]:
# Check order_date parsing failures
raw_dates = raw_df["order_date"]
parsed_dates = cleaned_df["order_date"]

bad_format_mask = raw_dates.notna() & parsed_dates.isna()
print(f"Unparseable order_date values: {bad_format_mask.sum()}")
print("Examples of unparseable values:")
print(raw_dates[bad_format_mask].head(20))

print("Samples of cleaned order_date values:")
print(parsed_dates.dropna().sample(100))

Unparseable order_date values: 0
Examples of unparseable values:
Series([], Name: order_date, dtype: str)
Samples of cleaned order_date values:
293611    2019-04-24
369882    2019-11-16
735588    2022-07-18
989342    2024-07-22
1125455   2025-12-09
             ...    
866910    2023-07-30
367575    2019-11-22
773868    2022-11-05
315198    2019-06-25
146504    2017-10-29
Name: order_date, Length: 100, dtype: datetime64[us]


In [18]:
# Clean original_price_inr to numeric INR
if "original_price_inr" not in cleaned_df.columns:
    raise KeyError("original_price_inr column not found in dataset.")

raw_price = cleaned_df["original_price_inr"].astype(str).str.strip()
is_texty = raw_price.str.contains(r"[^0-9\s,\.₹]", regex=True)

price_series = raw_price.str.replace(r"[₹,]", "", regex=True)
price_series = price_series.str.replace(r"[^0-9.]", "", regex=True)
price_series = price_series.mask(is_texty)
cleaned_df["original_price_inr"] = pd.to_numeric(price_series, errors="coerce")

price_invalid = cleaned_df["original_price_inr"].isna().sum()
print(f"Invalid original_price_inr entries coerced to NaN: {price_invalid}")

save_cleaned_df(cleaned_df, cleaned_dir, note="original_price_inr cleaned")

Invalid original_price_inr entries coerced to NaN: 36422
Skipped parquet export due to error: A type extension with name pandas.period already defined
Saved cleaned dataset to: C:\Users\admin\Desktop\Amazon_India_Sales_Analytics\data\cleaned\amazon_india_all_years_cleaned.csv
Checkpoint: original_price_inr cleaned


In [19]:
# Validate original_price_inr cleaning
raw_price_check = raw_df["original_price_inr"].astype(str).str.strip()
cleaned_price = cleaned_df["original_price_inr"]

non_numeric_raw = raw_price_check.str.contains(r"[^0-9\s,\.₹]", regex=True).sum()
print(f"Raw values with non-numeric text: {non_numeric_raw}")
print(f"Cleaned NaN count: {cleaned_price.isna().sum()}")
print(f"Cleaned dtype: {cleaned_price.dtype}")

print("Sample cleaned prices:")
print(cleaned_price.dropna().head(20))

Raw values with non-numeric text: 36422
Cleaned NaN count: 36422
Cleaned dtype: float64
Sample cleaned prices:
0     123614.29
1      54731.86
2      97644.25
3      21947.26
4      54731.86
5     131194.65
6      86987.64
7      32169.01
8      40264.16
9      54731.86
10     88664.85
11     73967.02
12     72564.10
13    209875.55
14     45363.47
15     23075.71
16     23075.71
17    114096.63
18     69584.33
19     38884.19
Name: original_price_inr, dtype: float64


In [20]:
# Standardize customer_rating to numeric 1.0-5.0
if "cleaned_df" not in globals():
    raise NameError("cleaned_df is not defined. Run the earlier load/clean cells first.")

if "customer_rating" not in cleaned_df.columns:
    raise KeyError("customer_rating column not found in dataset.")

rating_raw = cleaned_df["customer_rating"].astype(str).str.strip()

# Normalize common patterns
# Examples: "4 stars", "3/5", "2.5/5.0", "5.0"
rating_norm = (
    rating_raw
    .str.lower()
    .str.replace("stars", "", regex=False)
    .str.replace("star", "", regex=False)
    .str.replace("out of", "/", regex=False)
    .str.replace(r"\s+", "", regex=True)
    .str.replace(r"^none$|^nan$|^null$|^$", "", regex=True)
 )

# Extract numerator/denominator if in fraction form
fraction = rating_norm.str.extract(r"^(?P<num>\d+(\.\d+)?)/(?P<den>\d+(\.\d+)?)$")
num = pd.to_numeric(fraction["num"], errors="coerce")
den = pd.to_numeric(fraction["den"], errors="coerce")

# If fraction, convert to 1-5 scale
rating_fraction = (num / den) * 5

# If not fraction, try direct numeric
rating_direct = pd.to_numeric(rating_norm, errors="coerce")

# Combine: prefer fraction-derived if present, else direct
rating_clean = rating_fraction.combine_first(rating_direct)

# Clip to valid range and set out-of-range to NaN
rating_clean = rating_clean.where((rating_clean >= 1.0) & (rating_clean <= 5.0))

cleaned_df["customer_rating"] = rating_clean

rating_invalid = cleaned_df["customer_rating"].isna().sum()
print(f"Invalid/missing customer_rating entries: {rating_invalid}")
print("Sample cleaned ratings:")
print(cleaned_df["customer_rating"].dropna().head(20))

save_cleaned_df(cleaned_df, cleaned_dir, note="customer_rating cleaned")

Invalid/missing customer_rating entries: 341696
Sample cleaned ratings:
0     5.0
1     4.5
3     3.0
4     4.0
5     4.5
6     3.5
7     4.0
8     5.0
9     4.5
10    5.0
11    4.5
13    4.0
14    4.5
16    5.0
18    4.5
23    4.5
24    4.5
26    4.5
29    4.0
30    5.0
Name: customer_rating, dtype: float64
Skipped parquet export due to error: A type extension with name pandas.period already defined
Saved cleaned dataset to: C:\Users\admin\Desktop\Amazon_India_Sales_Analytics\data\cleaned\amazon_india_all_years_cleaned.csv
Checkpoint: customer_rating cleaned


In [27]:
# Explore current customer_city values (before standardization)
if "cleaned_df" not in globals():
    raise NameError("cleaned_df is not defined. Run the earlier load/clean cells first.")

if "customer_city" not in cleaned_df.columns:
    raise KeyError("customer_city column not found in dataset.")

city_series = cleaned_df["customer_city"].astype(str).str.strip()
city_series = city_series.replace({"": pd.NA, "nan": pd.NA, "none": pd.NA, "null": pd.NA})

print("Top 50 city values (raw):")
print(city_series.value_counts(dropna=False).tail(50))

print("Unique city count (non-null):", city_series.dropna().nunique())

print("Sample raw cities:")
print(city_series.dropna().sample(30, random_state=42))

Top 50 city values (raw):
customer_city
Mumbai           140840
Delhi            122365
Bangalore        101416
Chennai           83922
Pune              67703
Kolkata           66061
Ahmedabad         50861
Hyderabad         44573
Jaipur            40453
Surat             40400
Nagpur            37211
Kanpur            34334
Lucknow           34077
Indore            33592
Coimbatore        26695
Kochi             24940
Visakhapatnam     22167
Patna             21339
Vadodara          21078
Bhubaneswar       20472
Chandigarh        19575
Ludhiana          18891
Saharanpur         6705
Meerut             6641
Bareilly           6590
Aligarh            6407
Allahabad          6030
Varanasi           5897
Gorakhpur          5867
Moradabad          5855
New Delhi           304
Bombay              289
Banglore            287
KOLKATA             285
CHENNAI             285
delhi               283
Delhi NCR           282
chenai              280
mumba               279
BANGALORE           277


In [29]:
# Show sample corrections (raw -> standardized)
if "cleaned_df" not in globals():
    raise NameError("cleaned_df is not defined. Run the earlier load/clean cells first.")

if "customer_city" not in cleaned_df.columns:
    raise KeyError("customer_city column not found in dataset.")

sample_raw = cleaned_df["customer_city"].astype(str).str.strip()
sample_norm = (
    sample_raw
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.replace("/", " ", regex=False)
    .str.replace(r"[^a-z\s]", "", regex=True)
    .str.strip()
 )

sample_std = sample_norm.replace(city_aliases)
sample_std = sample_std.replace({"": pd.NA, "nan": pd.NA, "none": pd.NA, "null": pd.NA})
sample_std = sample_std.str.title()

sample_map = pd.DataFrame({
    "raw_city": sample_raw,
    "standardized_city": sample_std
})

changed = sample_map[
    sample_map["raw_city"].str.strip().str.lower()
    != sample_map["standardized_city"].fillna("").str.lower()
].dropna()

print("Sample corrections (raw -> standardized):")
print(changed.drop_duplicates().head(30))

Sample corrections (raw -> standardized):
Empty DataFrame
Columns: [raw_city, standardized_city]
Index: []


In [28]:
# Standardize customer_city names
if "cleaned_df" not in globals():
    raise NameError("cleaned_df is not defined. Run the earlier load/clean cells first.")

if "customer_city" not in cleaned_df.columns:
    raise KeyError("customer_city column not found in dataset.")

city_raw = cleaned_df["customer_city"].astype(str).str.strip()
city_norm = (
    city_raw
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.replace("/", " ", regex=False)
    .str.replace(r"[^a-z\s]", "", regex=True)
    .str.strip()
 )

city_aliases = {
    "bangalore": "Bengaluru",
    "bengaluru": "Bengaluru",
    "bengaluru bangalore": "Bengaluru",
    "bangalore bengaluru": "Bengaluru",
    "mumbai": "Mumbai",
    "bombay": "Mumbai",
    "mumbai bombay": "Mumbai",
    "bombay mumbai": "Mumbai",
    "delhi": "Delhi",
    "new delhi": "Delhi",
    "delhi new delhi": "Delhi",
    "new delhi delhi": "Delhi"
}

city_std = city_norm.replace(city_aliases)
city_std = city_std.replace({"": pd.NA, "nan": pd.NA, "none": pd.NA, "null": pd.NA})
city_std = city_std.str.title()
cleaned_df["customer_city"] = city_std

city_missing = cleaned_df["customer_city"].isna().sum()
print(f"Missing customer_city after standardization: {city_missing}")
print("Sample standardized cities:")
print(cleaned_df["customer_city"].dropna().head(20))

save_cleaned_df(cleaned_df, cleaned_dir, note="customer_city cleaned")

Missing customer_city after standardization: 0
Sample standardized cities:
0          Mumbai
1       Allahabad
2          Mumbai
3         Kolkata
4        Ludhiana
5           Delhi
6         Lucknow
7          Jaipur
8           Delhi
9     Bhubaneswar
10      Ahmedabad
11      Bengaluru
12          Delhi
13         Mumbai
14      Bengaluru
15           Pune
16          Kochi
17        Chennai
18        Chennai
19         Mumbai
Name: customer_city, dtype: str
Saved cleaned dataset to: C:\Users\admin\Desktop\Amazon_India_Sales_Analytics\data\cleaned\amazon_india_all_years_cleaned.csv
Checkpoint: customer_city cleaned


In [31]:
# Standardize boolean columns to True/False
if "cleaned_df" not in globals():
    raise NameError("cleaned_df is not defined. Run the earlier load/clean cells first.")

bool_columns = ["is_prime_member", "is_prime_eligible", "is_festival_sale"]
existing_bool_columns = [col for col in bool_columns if col in cleaned_df.columns]
missing_bool_columns = [col for col in bool_columns if col not in cleaned_df.columns]

if missing_bool_columns:
    print(f"Missing boolean columns (skipped): {missing_bool_columns}")

bool_map = {
    "true": True,
    "false": False,
    "yes": True,
    "no": False,
    "y": True,
    "n": False,
    "1": True,
    "0": False,
    "t": True,
    "f": False
}

for col in existing_bool_columns:
    raw_vals = cleaned_df[col].astype(str).str.strip().str.lower()
    raw_vals = raw_vals.replace({"": pd.NA, "nan": pd.NA, "none": pd.NA, "null": pd.NA})
    cleaned_df[col] = raw_vals.map(bool_map)

    invalid_count = cleaned_df[col].isna().sum()
    print(f"{col} invalid/missing after standardization: {invalid_count}")

save_cleaned_df(cleaned_df, cleaned_dir, note="boolean columns standardized")

is_prime_member invalid/missing after standardization: 0
is_prime_eligible invalid/missing after standardization: 0
is_festival_sale invalid/missing after standardization: 0
Saved cleaned dataset to: C:\Users\admin\Desktop\Amazon_India_Sales_Analytics\data\cleaned\amazon_india_all_years_cleaned.csv
Checkpoint: boolean columns standardized


In [32]:
# Explore category and subcategory values (cleaned all years)
if "cleaned_df" not in globals():
    raise NameError("cleaned_df is not defined. Run the earlier load/clean cells first.")

category_cols = [col for col in cleaned_df.columns if "category" in col.lower()]
print("Category-related columns:", category_cols)

for col in category_cols:
    series = cleaned_df[col].astype(str).str.strip()
    series = series.replace({"": pd.NA, "nan": pd.NA, "none": pd.NA, "null": pd.NA})
    print(f"\nTop 30 values for {col}:")
    print(series.value_counts(dropna=False).head(30))
    print(f"Unique {col} count (non-null):", series.dropna().nunique())

Category-related columns: ['category', 'subcategory']

Top 30 values for category:
category
Electronics                  1126726
Electronic                       229
ELECTRONICS                      225
Electronics & Accessories        218
Electronicss                     211
Name: count, dtype: int64
Unique category count (non-null): 5

Top 30 values for subcategory:
subcategory
Smartphones           827177
Laptops                88370
Smart Watch            74708
Tablets                70254
Audio                  50623
TV & Entertainment     16477
Name: count, dtype: int64
Unique subcategory count (non-null): 6


In [34]:
# Standardize category names (cleaned all years)
if "cleaned_df" not in globals():
    raise NameError("cleaned_df is not defined. Run the earlier load/clean cells first.")

category_col = "category"
if category_col not in cleaned_df.columns:
    raise KeyError("category column not found in dataset.")

cat_raw = cleaned_df[category_col].astype(str).str.strip()
cat_norm = (
    cat_raw
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.replace("&", "and", regex=False)
    .str.replace(r"[^a-z\s]", "", regex=True)
    .str.strip()
 )

category_aliases = {
    "electronics": "Electronics",
    "electronic": "Electronics",
    "electronicss": "Electronics",
    "electronics accessories": "Electronics",
    "electronics and accessories": "Electronics"
}

cat_std = cat_norm.replace(category_aliases)
cat_std = cat_std.replace({"": pd.NA, "nan": pd.NA, "none": pd.NA, "null": pd.NA})
cat_std = cat_std.fillna(cat_norm).str.title()
cleaned_df[category_col] = cat_std

print("Sample standardized categories:")
print(cleaned_df[category_col].dropna().head(20))

save_cleaned_df(cleaned_df, cleaned_dir, note="category cleaned")

Sample standardized categories:
0     Electronics
1     Electronics
2     Electronics
3     Electronics
4     Electronics
5     Electronics
6     Electronics
7     Electronics
8     Electronics
9     Electronics
10    Electronics
11    Electronics
12    Electronics
13    Electronics
14    Electronics
15    Electronics
16    Electronics
17    Electronics
18    Electronics
19    Electronics
Name: category, dtype: str
Saved cleaned dataset to: C:\Users\admin\Desktop\Amazon_India_Sales_Analytics\data\cleaned\amazon_india_all_years_cleaned.csv
Checkpoint: category cleaned


In [39]:
# Clean delivery_days to valid numeric values
if "cleaned_df" not in globals():
    raise NameError("cleaned_df is not defined. Run the earlier load/clean cells first.")

if "delivery_days" not in cleaned_df.columns:
    raise KeyError("delivery_days column not found in dataset.")

raw_days = cleaned_df["delivery_days"].astype(str).str.strip().str.lower()
raw_days = raw_days.replace({"": pd.NA, "nan": pd.NA, "none": pd.NA, "null": pd.NA})

# Handle common text patterns
raw_days = raw_days.replace({
    "same day": "0",
    "same-day": "0",
    "express": "1",
    "next day": "1",
    "next-day": "1",
    "-1": pd.NA
})

# Handle ranges like 1-2 days (take max)
range_match = raw_days.str.extract(r"(?P<min>\d+\.?\d*)\s*-\s*(?P<max>\d+\.?\d*)")
range_max = pd.to_numeric(range_match["max"], errors="coerce")

# Extract numeric values for non-range entries
num_lists = raw_days.str.findall(r"\d+\.?\d*")

def _range_to_value(nums):
    if not isinstance(nums, (list, tuple)) or len(nums) == 0:
        return pd.NA
    values = [float(x) for x in nums]
    return max(values)

cleaned_values = num_lists.apply(_range_to_value)
cleaned_values = pd.to_numeric(cleaned_values, errors="coerce")
cleaned_values = range_max.combine_first(cleaned_values)

# Remove invalid negatives and unrealistic values
cleaned_values = cleaned_values.where(cleaned_values >= 0)
cleaned_values = cleaned_values.where(cleaned_values <= 30)

cleaned_df["delivery_days"] = cleaned_values

invalid_count = cleaned_df["delivery_days"].isna().sum()
print(f"Invalid/missing delivery_days after cleaning: {invalid_count}")
print("Sample cleaned delivery_days:")
print(cleaned_df["delivery_days"].dropna().head(20))

save_cleaned_df(cleaned_df, cleaned_dir, note="delivery_days cleaned")

Invalid/missing delivery_days after cleaning: 6836
Sample cleaned delivery_days:
0     6.0
1     4.0
2     4.0
3     4.0
4     3.0
5     3.0
6     4.0
7     3.0
8     3.0
9     5.0
10    4.0
11    3.0
12    4.0
13    4.0
14    4.0
15    4.0
16    5.0
17    7.0
18    3.0
19    5.0
Name: delivery_days, dtype: float64
Saved cleaned dataset to: C:\Users\admin\Desktop\Amazon_India_Sales_Analytics\data\cleaned\amazon_india_all_years_cleaned.csv
Checkpoint: delivery_days cleaned


In [40]:
# Inspect delivery_days raw values
if "cleaned_df" not in globals():
    raise NameError("cleaned_df is not defined. Run the earlier load/clean cells first.")

if "delivery_days" not in cleaned_df.columns:
    raise KeyError("delivery_days column not found in dataset.")

delivery_raw = cleaned_df["delivery_days"].astype(str).str.strip()
delivery_raw = delivery_raw.replace({"": pd.NA, "nan": pd.NA, "none": pd.NA, "null": pd.NA})

print("Top 50 delivery_days values (raw):")
print(delivery_raw.value_counts(dropna=False).head(50))

print("Sample raw delivery_days:")
print(delivery_raw.dropna().sample(30, random_state=42))

Top 50 delivery_days values (raw):
delivery_days
3.0     288488
1.0     212779
4.0     206438
5.0     136608
2.0     130814
6.0     102668
7.0      34116
NaN       6836
0.0       6687
15.0      2175
Name: count, dtype: int64
Sample raw delivery_days:
54504      7.0
368844     4.0
896976     4.0
931800     2.0
334424     3.0
837934     1.0
706603     7.0
802320     6.0
366886     6.0
347285     4.0
250931     4.0
44857      6.0
1094739    3.0
261266     6.0
592531     1.0
897659     2.0
129501     4.0
373061     4.0
954344     2.0
712431     2.0
897726     6.0
484804     2.0
295270     2.0
1012714    4.0
1045690    3.0
815920     1.0
447839     4.0
968900     3.0
654802     3.0
252590     3.0
Name: delivery_days, dtype: str


In [41]:
# Inspect delivery_days min/max (numeric only)
if "cleaned_df" not in globals():
    raise NameError("cleaned_df is not defined. Run the earlier load/clean cells first.")

if "delivery_days" not in cleaned_df.columns:
    raise KeyError("delivery_days column not found in dataset.")

delivery_num = pd.to_numeric(cleaned_df["delivery_days"], errors="coerce")
print("Min delivery_days (numeric):", delivery_num.min())
print("Max delivery_days (numeric):", delivery_num.max())

Min delivery_days (numeric): 0.0
Max delivery_days (numeric): 15.0


In [42]:
# Inspect payment_method values
if "cleaned_df" not in globals():
    raise NameError("cleaned_df is not defined. Run the earlier load/clean cells first.")

payment_col = "payment_method"
if payment_col not in cleaned_df.columns:
    raise KeyError("payment_method column not found in dataset.")

payment_raw = cleaned_df[payment_col].astype(str).str.strip()
payment_raw = payment_raw.replace({"": pd.NA, "nan": pd.NA, "none": pd.NA, "null": pd.NA})

print("Top 50 payment_method values (raw):")
print(payment_raw.value_counts(dropna=False).head(50))

print("Unique payment_method count (non-null):", payment_raw.dropna().nunique())

print("Sample raw payment_method values:")
print(payment_raw.dropna().sample(30, random_state=42))

Top 50 payment_method values (raw):
payment_method
UPI            384228
COD            322831
Credit Card    172261
Debit Card     140202
Net Banking     64971
Wallet          22821
BNPL            20295
Name: count, dtype: int64
Unique payment_method count (non-null): 7
Sample raw payment_method values:
507695      Debit Card
57977              COD
611698             UPI
473805             UPI
977189             UPI
427011             COD
1080493            UPI
817142             UPI
308113             COD
666542             COD
858471             COD
789075             UPI
983605     Net Banking
736352             UPI
172305             COD
882538             UPI
761130      Debit Card
349083      Debit Card
297022     Credit Card
165724     Net Banking
309118             COD
553969             COD
156936             COD
1027750            UPI
308908             UPI
1066840     Debit Card
591561      Debit Card
403893      Debit Card
334831             UPI
738664             UPI
Nam

In [43]:
# Group payment_method to show all unique values with counts
if "cleaned_df" not in globals():
    raise NameError("cleaned_df is not defined. Run the earlier load/clean cells first.")

payment_col = "payment_method"
if payment_col not in cleaned_df.columns:
    raise KeyError("payment_method column not found in dataset.")

payment_raw = cleaned_df[payment_col].astype(str).str.strip()
payment_raw = payment_raw.replace({"": pd.NA, "nan": pd.NA, "none": pd.NA, "null": pd.NA})

payment_counts = payment_raw.value_counts(dropna=False).reset_index()
payment_counts.columns = [payment_col, "count"]

print("All unique payment_method values with counts:")
print(payment_counts)

All unique payment_method values with counts:
  payment_method   count
0            UPI  384228
1            COD  322831
2    Credit Card  172261
3     Debit Card  140202
4    Net Banking   64971
5         Wallet   22821
6           BNPL   20295


In [44]:
# Check missing payment_method count
if "cleaned_df" not in globals():
    raise NameError("cleaned_df is not defined. Run the earlier load/clean cells first.")

payment_col = "payment_method"
if payment_col not in cleaned_df.columns:
    raise KeyError("payment_method column not found in dataset.")

payment_missing = cleaned_df[payment_col].isna().sum()
print(f"Missing payment_method: {payment_missing}")

Missing payment_method: 0
